# R Jupyter user-library bootstrap

Install a requested CRAN package stack into a user-owned library on Linux. An existing R kernel, R build tools, native development libraries, and permitted repository access are required. This notebook does not install R, Jupyter, or operating-system packages.

1. Start a fresh **R** kernel. Review the user options at the top of the installer cell. The defaults request `tidyverse`, `drc`, `ggplot2`, `dplyr`, `scales`, `mgcv`, and `bit64`, install only missing or incompatible packages, and exclude direct Suggests.
2. Run the installer cell. It may download and compile packages and, when enabled and needed, download a checksum-verified CMake archive. Review the summary and `target-validation.csv`; requested target namespaces must load successfully in fresh R processes. Keep logs private: paths, repository URLs, and environment details can be sensitive.
3. Optionally run the final cell to make this run's deployment library available in future R sessions. It updates the home `.Renviron` only after successful target validation and a checked backup. Skip it when R startup is centrally managed. Restart the kernel after updating startup configuration.

Runtime packages and CMake retain their upstream licenses. Repository clearance and runtime software approval are separate responsibilities; see the README and release checklist. Clear all outputs before committing or sharing this notebook.


In [ ]:
# =====================================================================
# Durable CRAN dependency installer v6 for restricted Linux/Jupyter
# environments.
#
# Recommended use:
#   1. Start with install_policy = "missing-only". It reuses compatible
#      installed packages but upgrades a hard dependency when its version
#      is too old.
#   2. Run with include_direct_suggests = FALSE installs only the runtime 
#      stack.
#   3. Run again with include_direct_suggests = TRUE for optional packages.
#      Turning on could help get other useful packages, but can easily
#      balloon into far more libs than intended.
#
# This tool targets Linux and assumes Linux native-library layouts.
# Other platforms are rejected before directories or files are created.
# =====================================================================

# ---------------------------------------------------------------------
# USER OPTIONS
#
# Edit the values below. Leave a value as NULL to fall back to the
# matching environment variable, and if that is unset, the built-in
# default. An untouched notebook behaves exactly like the previous
# version. Environment variables still work as overrides for anyone
# who prefers them.
# ---------------------------------------------------------------------

user_options <- list(

  # Packages you want installed. This is the one setting almost everyone
  # edits. List the top-level packages; their dependencies are resolved
  # automatically.
  targets = c(
    "tidyverse",
    "drc",
    "ggplot2",
    "dplyr",
    "scales",
    "mgcv",
    "bit64"
  ),

  # Install strategy:
  #   "missing-only"        install only what is missing or version-incompatible
  #   "missing-or-outdated" also upgrade anything older than the repo version
  #   "all-to-deploy"       ensure repository-current packages in deploy_lib
  #                         (reuse compatible/current deploy copies)
  install_policy = NULL,          # default: "missing-only"

  # FALSE: install the runtime stack only.
  # TRUE:  also attempt each target's optional (Suggests) packages.
  include_direct_suggests = NULL, # default: FALSE

  # Destination library for installed packages.
  deploy_lib = NULL,              # default: ~/R/library/<R ver>/<platform>

  # CRAN mirror or internal repository URL.
  cran_repo = NULL,               # default: https://cloud.r-project.org

  # Build parallelism hint and per-package build timeout (seconds).
  install_ncpus = NULL,           # default: 2
  install_timeout_seconds = NULL, # default: 7200

  # CMake handling. If no usable CMake is found and auto-bootstrap is on,
  # an official prebuilt CMake is downloaded into the user tools dir.
  auto_bootstrap_cmake = NULL,    # default: TRUE
  cmake_bootstrap_version = NULL, # default: "4.4.2"

  # Path-separated list of extra prefixes to search for native libraries
  # (headers/.so/.pc). The main escape hatch when a native dependency
  # lives somewhere nonstandard.
  native_prefixes = NULL          # default: "" (use built-in prefix list)
)

# ---------------------------------------------------------------------
# OPTION RESOLVER  (no need to edit below this line)
#
# Precedence for every option: value set above  ->  environment variable
# ->  built-in default. Most users will only want toedit the options  
# above; inline values override the environment variables.
# ---------------------------------------------------------------------

# Reset on every run so the optional persistence cell cannot use an earlier
# successful run after a later installer error.
installer_completed <- FALSE
if (!identical(unname(Sys.info()[["sysname"]]), "Linux")) {
  stop("This notebook supports Linux only; no installation was attempted.")
}

options(stringsAsFactors = FALSE)

`%||%` <- function(x, y) {
  if (length(x) && !is.na(x[[1]]) && nzchar(x[[1]])) x else y
}

as_flag <- function(x, default = FALSE) {
  if (!length(x) || is.na(x) || !nzchar(x)) return(default)
  tolower(x) %in% c("1", "true", "yes", "y", "on")
}

# Resolve a scalar option: inline value wins; else env var; else default.
opt_get <- function(inline, env_name, default) {
  if (!is.null(inline) && length(inline) && !is.na(inline[[1]]) &&
      nzchar(as.character(inline[[1]]))) {
    return(as.character(inline[[1]]))
  }
  Sys.getenv(env_name, unset = default)
}

r_minor <- paste(
  R.version$major,
  strsplit(R.version$minor, ".", fixed = TRUE)[[1]][1],
  sep = "."
)

default_deploy_lib <- file.path(
  path.expand("~/R/library"),
  r_minor,
  R.version$platform
)

deploy_lib <- path.expand(
  opt_get(user_options$deploy_lib, "R_DEPLOY_LIB", default_deploy_lib)
)

include_direct_suggests <- as_flag(
  opt_get(user_options$include_direct_suggests, "INCLUDE_DIRECT_SUGGESTS", "false")
)

install_policy <- opt_get(
  user_options$install_policy, "INSTALL_POLICY", "missing-only"
)

if (!install_policy %in% c("missing-or-outdated", "all-to-deploy", "missing-only")) {
  stop("Unsupported install_policy: ", install_policy)
}

install_timeout <- suppressWarnings(as.integer(
  opt_get(user_options$install_timeout_seconds, "INSTALL_TIMEOUT_SECONDS", "7200")
))
if (is.na(install_timeout) || install_timeout < 1L) install_timeout <- 7200L

ncpus <- suppressWarnings(as.integer(
  opt_get(user_options$install_ncpus, "INSTALL_NCPUS", "2")
))
if (is.na(ncpus) || ncpus < 1L) ncpus <- 1L

repo <- opt_get(user_options$cran_repo, "CRAN_REPO", "https://cloud.r-project.org")
repos <- c(CRAN = repo)

# targets has no environment-variable equivalent in the original script,
# so it is taken directly from the user options above.
targets <- user_options$targets
stopifnot(is.character(targets), length(targets) >= 1L)


###### Main Script Start ########
run_id <- format(Sys.time(), "%Y%m%d-%H%M%S")
log_dir <- path.expand(file.path("~/R", paste0("deps-install-logs-", run_id)))
source_dir <- file.path(log_dir, "sources")
forensics_dir <- file.path(log_dir, "forensics")

dir.create(deploy_lib, recursive = TRUE, showWarnings = FALSE)
dir.create(log_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(source_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(forensics_dir, recursive = TRUE, showWarnings = FALSE)

if (file.access(deploy_lib, 2) != 0L) {
  stop("Deployment library is not writable: ", deploy_lib)
}

# Use an exact, clean search path. Do not append inherited user libraries,
# because stale packages in them can shadow the managed R library.
system_libs <- unique(c(.Library, .Library.site))
system_libs <- system_libs[nzchar(system_libs) & dir.exists(system_libs)]
search_libs <- unique(c(deploy_lib, system_libs))
.libPaths(search_libs)

options(
  repos = repos,
  warn = 1,
  Ncpus = ncpus
)

master_log <- file.path(log_dir, "master.log")
status_file <- file.path(log_dir, "status.csv")

mlog <- function(...) {
  line <- paste0(format(Sys.time(), "%Y-%m-%d %H:%M:%S "), paste0(..., collapse = ""))
  cat(line, "\n")
  cat(line, "\n", file = master_log, append = TRUE)
  flush.console()
}

safe_lines <- function(path) {
  if (!file.exists(path)) return(character())
  tryCatch(readLines(path, warn = FALSE), error = function(e) character())
}

strip_ansi <- function(x) {
  gsub("\\033\\[[0-9;?]*[ -/]*[@-~]", "", x, perl = TRUE)
}

pkg_record <- function(pkg, libs = search_libs) {
  for (lib in libs) {
    desc <- file.path(lib, pkg, "DESCRIPTION")
    if (file.exists(desc)) {
      d <- tryCatch(
        read.dcf(desc, fields = c("Package", "Version")),
        error = function(e) NULL
      )
      if (!is.null(d) && nrow(d)) {
        return(list(
          present = TRUE,
          version = unname(d[1, "Version"]),
          path = normalizePath(file.path(lib, pkg), mustWork = FALSE),
          lib = normalizePath(lib, mustWork = FALSE)
        ))
      }
    }
  }
  list(present = FALSE, version = NA_character_, path = NA_character_, lib = NA_character_)
}

version_at_least <- function(installed, required) {
  if (is.na(installed) || !nzchar(installed)) return(FALSE)
  if (is.na(required) || !nzchar(required)) return(TRUE)
  utils::compareVersion(installed, required) >= 0L
}

# Do not force /opt/conda/lib into LD_LIBRARY_PATH. In a Conda-built R it may
# cause an unrelated shared library to win over a package-local library.
build_env <- character()

split_path_list <- function(value) {
  if (!length(value) || is.na(value) || !nzchar(value)) return(character())
  out <- strsplit(value, .Platform$path.sep, fixed = TRUE)[[1]]
  unique(out[nzchar(out)])
}

# Native dependencies may live in the managed Conda prefix, a user-owned
# prefix, or conventional system prefixes. Prefer explicit/user prefixes and
# the active Conda prefix before falling back to /usr/local and /usr.
native_prefixes <- unique(c(
  split_path_list(
    opt_get(user_options$native_prefixes, "NATIVE_PREFIXES", "")
  ),
  Sys.getenv("CONDA_PREFIX", unset = ""),
  "/opt/conda",
  path.expand("~/R/native"),
  "/usr/local",
  "/usr"
))
           
native_prefixes <- native_prefixes[nzchar(native_prefixes) & dir.exists(native_prefixes)]

native_bin_dirs <- file.path(native_prefixes, "bin")
path_prepend <- native_bin_dirs[dir.exists(native_bin_dirs)]

# Several Anticonf configure scripts call the literal command `pkg-config`
# rather than honoring PKG_CONFIG. If only pkgconf is available, create a
# user-writable compatibility wrapper named pkg-config and put it on PATH.
pkg_config_candidates <- unique(c(
  Sys.getenv("PKG_CONFIG_BIN", unset = ""),
  unname(Sys.which("pkg-config")),
  unname(Sys.which("pkgconf")),
  unlist(lapply(native_prefixes, function(prefix) c(
    file.path(prefix, "bin", "pkg-config"),
    file.path(prefix, "bin", "pkgconf")
  )), use.names = FALSE),
  Sys.glob(path.expand("~/R/tools/pkgconf-*/bin/pkgconf")),
  Sys.glob(path.expand("~/R/tools/pkgconf-*/bin/pkg-config"))
))
pkg_config_candidates <- pkg_config_candidates[
  nzchar(pkg_config_candidates) &
    file.exists(pkg_config_candidates) &
    file.access(pkg_config_candidates, 1) == 0L
]

pkg_config <- ""
if (length(pkg_config_candidates)) {
  selected_pkg_config <- normalizePath(pkg_config_candidates[[1]], mustWork = TRUE)

  if (!identical(basename(selected_pkg_config), "pkg-config")) {
    wrapper_dir <- file.path(log_dir, "tools", "bin")
    dir.create(wrapper_dir, recursive = TRUE, showWarnings = FALSE)
    wrapper <- file.path(wrapper_dir, "pkg-config")
    writeLines(c(
      "#!/bin/sh",
      paste("exec", shQuote(selected_pkg_config), '"$@"')
    ), wrapper)
    Sys.chmod(wrapper, mode = "0755")
    pkg_config <- normalizePath(wrapper, mustWork = TRUE)
  } else {
    pkg_config <- selected_pkg_config
  }

  build_env["PKG_CONFIG"] <- pkg_config
  path_prepend <- c(dirname(pkg_config), path_prepend)
}

pkgconfig_dirs <- unique(c(
  unlist(lapply(native_prefixes, function(prefix) c(
    file.path(prefix, "lib", "pkgconfig"),
    file.path(prefix, "lib64", "pkgconfig"),
    file.path(prefix, "share", "pkgconfig"),
    file.path(prefix, "lib", "x86_64-linux-gnu", "pkgconfig")
  )), use.names = FALSE),
  "/usr/lib/x86_64-linux-gnu/pkgconfig"
))
pkgconfig_dirs <- pkgconfig_dirs[dir.exists(pkgconfig_dirs)]
if (length(pkgconfig_dirs)) {
  inherited <- split_path_list(Sys.getenv("PKG_CONFIG_PATH"))
  build_env["PKG_CONFIG_PATH"] <- paste(
    unique(c(pkgconfig_dirs, inherited)),
    collapse = .Platform$path.sep
  )
}

# Reuse an existing or user-local CMake without assuming a particular username.
# If no executable is available, bootstrap an official prebuilt CMake archive
# into the user's home directory. This keeps the install code-only and avoids
# requiring root access or a user-managed Conda environment.
#
# nloptr specifically honors CMAKE_BIN, whereas Arrow commonly honors CMAKE,
# so export the same absolute executable under both names.
auto_bootstrap_cmake <- as_flag(
  opt_get(user_options$auto_bootstrap_cmake, "AUTO_BOOTSTRAP_CMAKE", "true"),
  default = TRUE
)
cmake_bootstrap_version <- opt_get(
  user_options$cmake_bootstrap_version, "CMAKE_BOOTSTRAP_VERSION", "4.4.2"
)
cmake_tools_dir <- path.expand(Sys.getenv(
  "CMAKE_TOOLS_DIR",
  unset = "~/R/tools"
))

find_cmake_candidates <- function() {
  candidates <- unique(c(
    Sys.getenv("CMAKE_BIN", unset = ""),
    Sys.getenv("CMAKE", unset = ""),
    unname(Sys.which("cmake")),
    unname(Sys.which("cmake3")),
    Sys.glob(file.path(cmake_tools_dir, "cmake-*/bin/cmake")),
    Sys.glob(file.path(cmake_tools_dir, "cmake-*-linux-*/bin/cmake"))
  ))

  candidates <- candidates[
    nzchar(candidates) &
      file.exists(candidates) &
      file.access(candidates, 1) == 0L
  ]

  if (!length(candidates)) return(character())

  valid <- vapply(candidates, function(candidate) {
    identical(
      suppressWarnings(system2(
        candidate,
        "--version",
        stdout = FALSE,
        stderr = FALSE
      )),
      0L
    )
  }, logical(1))

  unique(normalizePath(candidates[valid], mustWork = TRUE))
}

cmake_bootstrap_record <- data.frame(
  attempted = FALSE,
  success = FALSE,
  version = cmake_bootstrap_version,
  platform = unname(Sys.info()[["sysname"]] %||% ""),
  machine = unname(Sys.info()[["machine"]] %||% ""),
  url = "",
  archive = "",
  executable = "",
  checksum_expected = "",
  checksum_actual = "",
  checksum_verified = FALSE,
  message = "not attempted",
  stringsAsFactors = FALSE
)

sha256_file <- function(path) {
  sha256sum <- unname(Sys.which("sha256sum"))
  if (nzchar(sha256sum)) {
    out <- system2(sha256sum, shQuote(path), stdout = TRUE, stderr = TRUE)
    status <- attr(out, "status") %||% 0L
    if (identical(as.integer(status), 0L) && length(out)) {
      return(tolower(strsplit(trimws(out[[1]]), "[[:space:]]+")[[1]][[1]]))
    }
  }

  shasum <- unname(Sys.which("shasum"))
  if (nzchar(shasum)) {
    out <- system2(shasum, c("-a", "256", shQuote(path)), stdout = TRUE, stderr = TRUE)
    status <- attr(out, "status") %||% 0L
    if (identical(as.integer(status), 0L) && length(out)) {
      return(tolower(strsplit(trimws(out[[1]]), "[[:space:]]+")[[1]][[1]]))
    }
  }

  ""
}

bootstrap_cmake <- function(version, tools_dir) {
  sysname <- unname(Sys.info()[["sysname"]] %||% "")
  machine <- tolower(unname(Sys.info()[["machine"]] %||% ""))

  if (!identical(sysname, "Linux")) {
    return(list(
      success = FALSE,
      message = paste("automatic CMake bootstrap currently supports Linux only; found", sysname)
    ))
  }

  asset_arch <- switch(
    machine,
    "x86_64" = "x86_64",
    "amd64" = "x86_64",
    "aarch64" = "aarch64",
    "arm64" = "aarch64",
    ""
  )

  if (!nzchar(asset_arch)) {
    return(list(
      success = FALSE,
      message = paste("unsupported Linux architecture for CMake bootstrap:", machine)
    ))
  }

  dir.create(tools_dir, recursive = TRUE, showWarnings = FALSE)
  downloads_dir <- file.path(tools_dir, "downloads")
  dir.create(downloads_dir, recursive = TRUE, showWarnings = FALSE)

  cmake_name <- paste0("cmake-", version, "-linux-", asset_arch)
  archive_name <- paste0(cmake_name, ".tar.gz")
  install_dir <- file.path(tools_dir, cmake_name)
  executable <- file.path(install_dir, "bin", "cmake")
  base_url <- paste0(
    "https://github.com/Kitware/CMake/releases/download/v",
    version
  )
  archive_url <- paste0(base_url, "/", archive_name)
  checksum_url <- paste0(
    base_url,
    "/cmake-", version, "-SHA-256.txt"
  )
  archive_path <- file.path(downloads_dir, archive_name)
  checksum_path <- file.path(
    downloads_dir,
    paste0("cmake-", version, "-SHA-256.txt")
  )

  if (file.exists(executable) && file.access(executable, 1) == 0L) {
    status <- suppressWarnings(system2(
      executable,
      "--version",
      stdout = FALSE,
      stderr = FALSE
    ))
    if (identical(as.integer(status), 0L)) {
      return(list(
        success = TRUE,
        message = "existing bootstrapped CMake is usable",
        url = archive_url,
        archive = archive_path,
        executable = normalizePath(executable, mustWork = TRUE),
        checksum_expected = "",
        checksum_actual = "",
        checksum_verified = NA
      ))
    }
  }

  download_one <- function(url, dest) {
    tmp <- paste0(dest, ".part")
    unlink(tmp, force = TRUE)
    utils::download.file(
      url,
      destfile = tmp,
      mode = "wb",
      quiet = FALSE
    )
    if (!file.exists(tmp) || file.info(tmp)$size <= 0) {
      stop("download produced no file: ", url)
    }
    if (file.exists(dest)) unlink(dest, force = TRUE)
    if (!file.rename(tmp, dest)) {
      unlink(tmp, force = TRUE)
      stop("could not move downloaded file into place: ", dest)
    }
    dest
  }

  result <- tryCatch({
    if (!file.exists(archive_path) || file.info(archive_path)$size <= 0) {
      download_one(archive_url, archive_path)
    }
    download_one(checksum_url, checksum_path)

    checksum_lines <- readLines(checksum_path, warn = FALSE)
    checksum_fields <- strsplit(trimws(checksum_lines), "[[:space:]]+")
    checksum_match <- vapply(checksum_fields, function(parts) {
      length(parts) >= 2L &&
        identical(sub("^\\*", "", tail(parts, 1L)), archive_name) &&
        grepl("^[0-9A-Fa-f]{64}$", parts[[1]])
    }, logical(1))
    if (!any(checksum_match)) {
      stop("official checksum file did not contain an entry for ", archive_name)
    }

    expected <- tolower(checksum_fields[[which(checksum_match)[[1]]]][[1]])
    actual <- sha256_file(archive_path)
    if (!nzchar(actual)) {
      stop("neither sha256sum nor shasum was available to verify CMake")
    }
    if (!identical(actual, expected)) {
      stop(
        "CMake archive SHA-256 mismatch: expected ", expected,
        ", got ", actual
      )
    }

    if (dir.exists(install_dir)) {
      unlink(install_dir, recursive = TRUE, force = TRUE)
    }
    utils::untar(archive_path, exdir = tools_dir)

    if (!file.exists(executable)) {
      stop("CMake archive extracted but executable was not found: ", executable)
    }
    Sys.chmod(executable, mode = "0755")
    status <- suppressWarnings(system2(
      executable,
      "--version",
      stdout = FALSE,
      stderr = FALSE
    ))
    if (!identical(as.integer(status), 0L)) {
      stop("bootstrapped CMake executable did not run successfully")
    }

    list(
      success = TRUE,
      message = "downloaded, checksum-verified, and extracted official CMake",
      url = archive_url,
      archive = normalizePath(archive_path, mustWork = TRUE),
      executable = normalizePath(executable, mustWork = TRUE),
      checksum_expected = expected,
      checksum_actual = actual,
      checksum_verified = TRUE
    )
  }, error = function(e) {
    list(
      success = FALSE,
      message = conditionMessage(e),
      url = archive_url,
      archive = archive_path,
      executable = executable,
      checksum_expected = "",
      checksum_actual = "",
      checksum_verified = FALSE
    )
  })

  result
}

cmake_candidates <- find_cmake_candidates()

if (!length(cmake_candidates) && auto_bootstrap_cmake) {
  cmake_bootstrap_record$attempted <- TRUE
  mlog(
    "CMake was not found; bootstrapping CMake ",
    cmake_bootstrap_version,
    " into ",
    cmake_tools_dir,
    "..."
  )

  boot <- bootstrap_cmake(cmake_bootstrap_version, cmake_tools_dir)
  cmake_bootstrap_record$success <- isTRUE(boot$success)
  cmake_bootstrap_record$url <- boot$url %||% ""
  cmake_bootstrap_record$archive <- boot$archive %||% ""
  cmake_bootstrap_record$executable <- boot$executable %||% ""
  cmake_bootstrap_record$checksum_expected <- boot$checksum_expected %||% ""
  cmake_bootstrap_record$checksum_actual <- boot$checksum_actual %||% ""
  cmake_bootstrap_record$checksum_verified <- isTRUE(boot$checksum_verified)
  cmake_bootstrap_record$message <- boot$message %||% ""

  if (isTRUE(boot$success)) {
    mlog("CMake bootstrap OK: ", boot$executable)
  } else {
    mlog("CMake bootstrap FAILED: ", boot$message)
  }

  cmake_candidates <- find_cmake_candidates()
}

if (length(cmake_candidates)) {
  # Candidates are ordered explicit CMAKE_BIN/CMAKE first, then PATH and
  # user-local discoveries. Keep that precedence when selecting the tool.
  cmake_bin <- normalizePath(cmake_candidates[[1]], mustWork = TRUE)
  build_env["CMAKE"] <- cmake_bin
  build_env["CMAKE_BIN"] <- cmake_bin
  path_prepend <- c(dirname(cmake_bin), path_prepend)
}

write.csv(
  cmake_bootstrap_record,
  file.path(log_dir, "cmake-bootstrap.csv"),
  row.names = FALSE
)

if (length(path_prepend)) {
  build_env["PATH"] <- paste(
    unique(c(path_prepend, split_path_list(Sys.getenv("PATH")))),
    collapse = .Platform$path.sep
  )
}

build_env["MAKEFLAGS"] <- paste0("-j", ncpus)
build_env["CMAKE_BUILD_PARALLEL_LEVEL"] <- as.character(ncpus)

# Only set udunits hints when both a header and library really exist.
ud_header_dirs <- c("/opt/conda/include", "/usr/local/include", "/usr/include")
ud_header_dir <- ud_header_dirs[vapply(
  ud_header_dirs,
  function(d) {
    file.exists(file.path(d, "udunits2.h")) ||
      file.exists(file.path(d, "udunits2", "udunits2.h"))
  },
  logical(1)
)]

ud_lib_dirs <- c("/opt/conda/lib", "/usr/local/lib64", "/usr/local/lib", "/usr/lib64", "/usr/lib")
ud_lib_dir <- ud_lib_dirs[vapply(
  ud_lib_dirs,
  function(d) length(Sys.glob(file.path(d, "libudunits2.so*"))) > 0L,
  logical(1)
)]

if (length(ud_header_dir) && length(ud_lib_dir)) {
  build_env["UDUNITS2_INCLUDE"] <- ud_header_dir[[1]]
  # The units configure script expects a DIRECTORY here, not "-L... -ludunits2".
  build_env["UDUNITS2_LIBS"] <- ud_lib_dir[[1]]
}

first_existing_rel <- function(prefix, relative_paths) {
  candidates <- file.path(prefix, relative_paths)
  hits <- candidates[file.exists(candidates)]
  if (length(hits)) normalizePath(hits[[1]], mustWork = TRUE) else ""
}

first_glob_rel <- function(prefix, relative_patterns) {
  hits <- unique(unlist(lapply(
    relative_patterns,
    function(pattern) Sys.glob(file.path(prefix, pattern))
  ), use.names = FALSE))
  hits <- hits[file.exists(hits)]
  if (length(hits)) normalizePath(hits[[1]], mustWork = TRUE) else ""
}

anticonf_path_value <- function(directories, flag) {
  directories <- unique(directories[nzchar(directories) & dir.exists(directories)])
  if (!length(directories)) return("")
  directories <- normalizePath(directories, mustWork = TRUE)
  paste(
    c(directories[[1]], if (length(directories) > 1L) paste0(flag, directories[-1]) else character()),
    collapse = " "
  )
}

# Find a complete libxml2 development bundle in one prefix. The xml2 package
# needs an include directory where <libxml/tree.h> resolves.
xml2_native <- NULL
for (prefix in native_prefixes) {
  header <- first_existing_rel(prefix, c(
    "include/libxml2/libxml/tree.h",
    "include/libxml/tree.h"
  ))
  library <- first_glob_rel(prefix, c(
    "lib/libxml2.so*", "lib64/libxml2.so*",
    "lib/x86_64-linux-gnu/libxml2.so*",
    "lib/libxml2.a", "lib64/libxml2.a",
    "lib/x86_64-linux-gnu/libxml2.a"
  ))
  if (nzchar(header) && nzchar(library)) {
    include_dir <- dirname(dirname(header))
    xml2_native <- list(
      prefix = normalizePath(prefix, mustWork = TRUE),
      header = header,
      library = library,
      include_dir = include_dir,
      lib_dir = dirname(library)
    )
    break
  }
}

if (!is.null(xml2_native)) {
  build_env["XML2_INCLUDE_DIR"] <- xml2_native$include_dir
  build_env["XML2_LIB_DIR"] <- xml2_native$lib_dir
}

# Find all native headers/libraries needed directly by ragg in one prefix.
# The manual Anticonf fallback accepts one INCLUDE_DIR/LIB_DIR string; extra
# directories are encoded as additional -I/-L tokens in those values.
ragg_native <- NULL
for (prefix in native_prefixes) {
  headers <- c(
    freetype = first_existing_rel(prefix, c(
      "include/freetype2/ft2build.h", "include/ft2build.h"
    )),
    png = first_existing_rel(prefix, c(
      "include/png.h", "include/libpng16/png.h"
    )),
    tiff = first_existing_rel(prefix, c(
      "include/tiffio.h", "include/libtiff/tiffio.h"
    )),
    jpeg = first_existing_rel(prefix, "include/jpeglib.h"),
    webp = first_existing_rel(prefix, "include/webp/encode.h")
  )

  libraries <- c(
    freetype = first_glob_rel(prefix, c(
      "lib/libfreetype.so*", "lib64/libfreetype.so*",
      "lib/x86_64-linux-gnu/libfreetype.so*", "lib/libfreetype.a"
    )),
    png = first_glob_rel(prefix, c(
      "lib/libpng16.so*", "lib/libpng.so*", "lib64/libpng16.so*",
      "lib/x86_64-linux-gnu/libpng16.so*", "lib/libpng16.a", "lib/libpng.a"
    )),
    tiff = first_glob_rel(prefix, c(
      "lib/libtiff.so*", "lib64/libtiff.so*",
      "lib/x86_64-linux-gnu/libtiff.so*", "lib/libtiff.a"
    )),
    jpeg = first_glob_rel(prefix, c(
      "lib/libjpeg.so*", "lib64/libjpeg.so*",
      "lib/x86_64-linux-gnu/libjpeg.so*", "lib/libjpeg.a"
    )),
    webp = first_glob_rel(prefix, c(
      "lib/libwebp.so*", "lib64/libwebp.so*",
      "lib/x86_64-linux-gnu/libwebp.so*", "lib/libwebp.a"
    )),
    webpmux = first_glob_rel(prefix, c(
      "lib/libwebpmux.so*", "lib64/libwebpmux.so*",
      "lib/x86_64-linux-gnu/libwebpmux.so*", "lib/libwebpmux.a"
    )),
    zlib = first_glob_rel(prefix, c(
      "lib/libz.so*", "lib64/libz.so*",
      "lib/x86_64-linux-gnu/libz.so*", "lib/libz.a"
    )),
    bzip2 = first_glob_rel(prefix, c(
      "lib/libbz2.so*", "lib64/libbz2.so*",
      "lib/x86_64-linux-gnu/libbz2.so*", "lib/libbz2.a"
    ))
  )

  if (all(nzchar(headers)) && all(nzchar(libraries))) {
    include_dirs <- c(
      dirname(headers[["freetype"]]),
      dirname(headers[["png"]]),
      dirname(headers[["tiff"]]),
      dirname(headers[["jpeg"]]),
      dirname(dirname(headers[["webp"]]))
    )
    lib_dirs <- dirname(unname(libraries))

    ragg_native <- list(
      prefix = normalizePath(prefix, mustWork = TRUE),
      headers = headers,
      libraries = libraries,
      include_value = anticonf_path_value(include_dirs, "-I"),
      lib_value = anticonf_path_value(lib_dirs, "-L")
    )
    break
  }
}

if (!is.null(ragg_native)) {
  build_env["RAGG_INCLUDE_DIR"] <- ragg_native$include_value
  build_env["RAGG_LIB_DIR"] <- ragg_native$lib_value
}

# Emit a detailed native dependency inventory even when a complete bundle is
# not available. This makes it clear whether the missing piece is an executable,
# a .pc metadata file, a header, or a link library.
native_scan_specs <- list(
  xml2_header = list(kind = "file", values = c(
    "include/libxml2/libxml/tree.h", "include/libxml/tree.h"
  )),
  xml2_library = list(kind = "glob", values = c(
    "lib/libxml2.so*", "lib64/libxml2.so*",
    "lib/x86_64-linux-gnu/libxml2.so*", "lib/libxml2.a"
  )),
  xml2_pc = list(kind = "file", values = c(
    "lib/pkgconfig/libxml-2.0.pc", "lib64/pkgconfig/libxml-2.0.pc",
    "lib/x86_64-linux-gnu/pkgconfig/libxml-2.0.pc",
    "share/pkgconfig/libxml-2.0.pc"
  )),
  freetype_header = list(kind = "file", values = c(
    "include/freetype2/ft2build.h", "include/ft2build.h"
  )),
  freetype_library = list(kind = "glob", values = c(
    "lib/libfreetype.so*", "lib64/libfreetype.so*",
    "lib/x86_64-linux-gnu/libfreetype.so*", "lib/libfreetype.a"
  )),
  freetype_pc = list(kind = "file", values = c(
    "lib/pkgconfig/freetype2.pc", "lib64/pkgconfig/freetype2.pc",
    "lib/x86_64-linux-gnu/pkgconfig/freetype2.pc",
    "share/pkgconfig/freetype2.pc"
  )),
  png_header = list(kind = "file", values = c(
    "include/png.h", "include/libpng16/png.h"
  )),
  png_library = list(kind = "glob", values = c(
    "lib/libpng16.so*", "lib/libpng.so*", "lib64/libpng16.so*",
    "lib/x86_64-linux-gnu/libpng16.so*", "lib/libpng16.a"
  )),
  png_pc = list(kind = "file", values = c(
    "lib/pkgconfig/libpng.pc", "lib/pkgconfig/libpng16.pc",
    "lib64/pkgconfig/libpng.pc", "lib64/pkgconfig/libpng16.pc",
    "lib/x86_64-linux-gnu/pkgconfig/libpng.pc",
    "share/pkgconfig/libpng.pc"
  )),
  tiff_header = list(kind = "file", values = c(
    "include/tiffio.h", "include/libtiff/tiffio.h"
  )),
  tiff_library = list(kind = "glob", values = c(
    "lib/libtiff.so*", "lib64/libtiff.so*",
    "lib/x86_64-linux-gnu/libtiff.so*", "lib/libtiff.a"
  )),
  tiff_pc = list(kind = "file", values = c(
    "lib/pkgconfig/libtiff-4.pc", "lib64/pkgconfig/libtiff-4.pc",
    "lib/x86_64-linux-gnu/pkgconfig/libtiff-4.pc",
    "share/pkgconfig/libtiff-4.pc"
  )),
  jpeg_header = list(kind = "file", values = "include/jpeglib.h"),
  jpeg_library = list(kind = "glob", values = c(
    "lib/libjpeg.so*", "lib64/libjpeg.so*",
    "lib/x86_64-linux-gnu/libjpeg.so*", "lib/libjpeg.a"
  )),
  jpeg_pc = list(kind = "file", values = c(
    "lib/pkgconfig/libjpeg.pc", "lib64/pkgconfig/libjpeg.pc",
    "lib/x86_64-linux-gnu/pkgconfig/libjpeg.pc",
    "share/pkgconfig/libjpeg.pc"
  )),
  webp_header = list(kind = "file", values = "include/webp/encode.h"),
  webp_library = list(kind = "glob", values = c(
    "lib/libwebp.so*", "lib64/libwebp.so*",
    "lib/x86_64-linux-gnu/libwebp.so*", "lib/libwebp.a"
  )),
  webp_pc = list(kind = "file", values = c(
    "lib/pkgconfig/libwebp.pc", "lib64/pkgconfig/libwebp.pc",
    "lib/x86_64-linux-gnu/pkgconfig/libwebp.pc",
    "share/pkgconfig/libwebp.pc"
  )),
  webpmux_library = list(kind = "glob", values = c(
    "lib/libwebpmux.so*", "lib64/libwebpmux.so*",
    "lib/x86_64-linux-gnu/libwebpmux.so*", "lib/libwebpmux.a"
  )),
  webpmux_pc = list(kind = "file", values = c(
    "lib/pkgconfig/libwebpmux.pc", "lib64/pkgconfig/libwebpmux.pc",
    "lib/x86_64-linux-gnu/pkgconfig/libwebpmux.pc",
    "share/pkgconfig/libwebpmux.pc"
  )),
  zlib_library = list(kind = "glob", values = c(
    "lib/libz.so*", "lib64/libz.so*",
    "lib/x86_64-linux-gnu/libz.so*", "lib/libz.a"
  )),
  zlib_pc = list(kind = "file", values = c(
    "lib/pkgconfig/zlib.pc", "lib64/pkgconfig/zlib.pc",
    "lib/x86_64-linux-gnu/pkgconfig/zlib.pc",
    "share/pkgconfig/zlib.pc"
  )),
  bzip2_library = list(kind = "glob", values = c(
    "lib/libbz2.so*", "lib64/libbz2.so*",
    "lib/x86_64-linux-gnu/libbz2.so*", "lib/libbz2.a"
  ))
)

native_scan <- do.call(rbind, lapply(native_prefixes, function(prefix) {
  do.call(rbind, lapply(names(native_scan_specs), function(requirement) {
    spec <- native_scan_specs[[requirement]]
    path <- if (identical(spec$kind, "file")) {
      first_existing_rel(prefix, spec$values)
    } else {
      first_glob_rel(prefix, spec$values)
    }
    data.frame(
      prefix = normalizePath(prefix, mustWork = TRUE),
      requirement = requirement,
      found = nzchar(path),
      path = path,
      stringsAsFactors = FALSE
    )
  }))
}))
write.csv(
  native_scan,
  file.path(log_dir, "native-dependency-scan.csv"),
  row.names = FALSE
)

pc_modules <- c(
  "libxml-2.0", "freetype2", "libpng", "libtiff-4",
  "libjpeg", "libwebp", "libwebpmux", "zlib"
)
pc_module_report <- data.frame(
  module = pc_modules,
  available = FALSE,
  cflags = "",
  libs = "",
  stringsAsFactors = FALSE
)
if (nzchar(pkg_config)) {
  pc_env <- if ("PKG_CONFIG_PATH" %in% names(build_env)) {
    paste0("PKG_CONFIG_PATH=", shQuote(unname(build_env[["PKG_CONFIG_PATH"]])))
  } else {
    character()
  }

  for (i in seq_along(pc_modules)) {
    module <- pc_modules[[i]]
    status <- suppressWarnings(system2(
      pkg_config,
      c("--exists", module),
      stdout = FALSE,
      stderr = FALSE,
      env = pc_env
    ))
    pc_module_report$available[[i]] <- identical(as.integer(status), 0L)
    if (pc_module_report$available[[i]]) {
      pc_module_report$cflags[[i]] <- paste(system2(
        pkg_config, c("--cflags", module), stdout = TRUE, stderr = TRUE, env = pc_env
      ), collapse = " ")
      pc_module_report$libs[[i]] <- paste(system2(
        pkg_config, c("--libs", module), stdout = TRUE, stderr = TRUE, env = pc_env
      ), collapse = " ")
    }
  }
}
write.csv(
  pc_module_report,
  file.path(log_dir, "pkg-config-modules.csv"),
  row.names = FALSE
)

# Record the environment before changing anything in child processes.
writeLines(capture.output({
  cat("R version\n")
  print(R.version.string)
  cat("\nPlatform\n")
  print(R.version$platform)
  cat("\nLibrary paths\n")
  print(.libPaths())
  cat("\nRepositories\n")
  print(getOption("repos"))
  cat("\nTargets\n")
  print(targets)
  cat("\nInclude direct Suggests\n")
  print(include_direct_suggests)
  cat("\nInstall policy\n")
  print(install_policy)
  cat("\nRelevant inherited environment\n")
  print(Sys.getenv(c(
    "PATH", "LD_LIBRARY_PATH", "LIBRARY_PATH", "CPATH",
    "PKG_CONFIG_PATH", "R_LIBS", "R_LIBS_USER", "R_LIBS_SITE",
    "R_MAKEVARS_USER", "MAKEFLAGS"
  )))
  cat("\nNative prefixes searched\n")
  print(native_prefixes)
  cat("\nChild build environment overrides\n")
  print(build_env)
  cat("\nBuild tools\n")
  print(Sys.which(c(
    "gcc", "g++", "gfortran", "make", "cmake", "pkg-config", "pkgconf",
    "ld", "readelf", "ldd"
  )))
  cat("\nSession info\n")
  print(sessionInfo())
}), file.path(log_dir, "environment.txt"))

sysreq <- data.frame(
  requirement = c(
    "udunits2_header", "udunits2_library", "pkg_config", "cmake",
    "gcc", "gxx", "gfortran", "make", "gdal_config", "geos_config", "proj"
  ),
  found = c(
    length(ud_header_dir) > 0L,
    length(ud_lib_dir) > 0L,
    nzchar(pkg_config),
    length(cmake_candidates) > 0L,
    nzchar(Sys.which("gcc")),
    nzchar(Sys.which("g++")),
    nzchar(Sys.which("gfortran")),
    nzchar(Sys.which("make")),
    nzchar(Sys.which("gdal-config")),
    nzchar(Sys.which("geos-config")),
    nzchar(Sys.which("proj"))
  ),
  detail = c(
    paste(ud_header_dir, collapse = ";"),
    paste(ud_lib_dir, collapse = ";"),
    pkg_config,
    paste(cmake_candidates, collapse = ";"),
    unname(Sys.which("gcc")),
    unname(Sys.which("g++")),
    unname(Sys.which("gfortran")),
    unname(Sys.which("make")),
    unname(Sys.which("gdal-config")),
    unname(Sys.which("geos-config")),
    unname(Sys.which("proj"))
  ),
  stringsAsFactors = FALSE
)
native_sysreq <- data.frame(
  requirement = c(
    "xml2_native_bundle", "ragg_native_bundle",
    "xml2_header", "xml2_library",
    "ragg_headers", "ragg_libraries"
  ),
  found = c(
    !is.null(xml2_native), !is.null(ragg_native),
    !is.null(xml2_native) && nzchar(xml2_native$header),
    !is.null(xml2_native) && nzchar(xml2_native$library),
    !is.null(ragg_native) && all(nzchar(ragg_native$headers)),
    !is.null(ragg_native) && all(nzchar(ragg_native$libraries))
  ),
  detail = c(
    if (is.null(xml2_native)) "" else xml2_native$prefix,
    if (is.null(ragg_native)) "" else ragg_native$prefix,
    if (is.null(xml2_native)) "" else xml2_native$header,
    if (is.null(xml2_native)) "" else xml2_native$library,
    if (is.null(ragg_native)) "" else paste(ragg_native$headers, collapse = ";"),
    if (is.null(ragg_native)) "" else paste(ragg_native$libraries, collapse = ";")
  ),
  stringsAsFactors = FALSE
)
sysreq <- rbind(sysreq, native_sysreq)
write.csv(sysreq, file.path(log_dir, "system-requirements.csv"), row.names = FALSE)

mlog("Resolving repository metadata...")
available <- utils::available.packages(repos = repos)
saveRDS(available, file.path(log_dir, "available-packages.rds"))
write.csv(available, file.path(log_dir, "available-packages.csv"), row.names = TRUE)

hard_types <- c("Depends", "Imports", "LinkingTo")
clean_names <- function(x) {
  x <- unique(as.character(x))
  x[!is.na(x) & nzchar(x) & x != "R"]
}

mlog("Resolving recursive hard dependency closure...")
hard_map <- tools::package_dependencies(
  targets,
  db = available,
  which = hard_types,
  recursive = TRUE
)
hard_pkgs <- clean_names(unlist(hard_map, use.names = FALSE))

suggest_pkgs <- character()
suggest_hard_pkgs <- character()
if (include_direct_suggests) {
  mlog("Adding targets' direct Suggests and those packages' recursive hard dependencies...")
  suggest_map <- tools::package_dependencies(
    targets,
    db = available,
    which = "Suggests",
    recursive = FALSE
  )
  suggest_pkgs <- clean_names(unlist(suggest_map, use.names = FALSE))

  if (length(suggest_pkgs)) {
    suggest_hard_map <- tools::package_dependencies(
      suggest_pkgs,
      db = available,
      which = hard_types,
      recursive = TRUE
    )
    suggest_hard_pkgs <- clean_names(unlist(suggest_hard_map, use.names = FALSE))
  }
}

base_recommended <- rownames(installed.packages(
  lib.loc = system_libs,
  priority = c("base", "recommended")
))
base_recommended <- unique(base_recommended)

requested_closure <- clean_names(c(
  targets,
  hard_pkgs,
  suggest_pkgs,
  suggest_hard_pkgs
))

base_requested <- intersect(requested_closure, base_recommended)
unavailable <- setdiff(requested_closure, c(rownames(available), base_recommended))
all_pkgs <- intersect(setdiff(requested_closure, base_recommended), rownames(available))

writeLines(sort(base_recommended), file.path(log_dir, "base-recommended-skipped.txt"))
writeLines(sort(base_requested), file.path(log_dir, "base-recommended-requested.txt"))
writeLines(sort(unavailable), file.path(log_dir, "closure-unavailable.txt"))
writeLines(sort(all_pkgs), file.path(log_dir, "closure-all.txt"))

roles <- setNames(rep("suggest-hard", length(all_pkgs)), all_pkgs)
roles[intersect(all_pkgs, suggest_pkgs)] <- "direct-suggest"
roles[intersect(all_pkgs, hard_pkgs)] <- "hard"
roles[intersect(all_pkgs, targets)] <- "target"

# Direct hard edges are needed both for a topological install order and for
# marking packages BLOCKED when a prerequisite could not be installed.
direct_hard <- tools::package_dependencies(
  all_pkgs,
  db = available,
  which = hard_types,
  recursive = FALSE
)

direct_hard <- lapply(all_pkgs, function(p) {
  deps <- clean_names(direct_hard[[p]])
  intersect(setdiff(deps, base_recommended), all_pkgs)
})
names(direct_hard) <- all_pkgs

# tools::package_dependencies() intentionally returns package names only.  For
# an installation policy such as "missing-only", package presence is not enough:
# every installed hard dependency must also satisfy the version predicates in
# Depends, Imports, and LinkingTo.  Keep those predicates separately so an old
# managed package can be upgraded only when compatibility actually requires it.
split_dependency_spec <- function(x) {
  x <- trimws(x)
  if (!nzchar(x)) return(NULL)

  pattern <- paste0(
    "^([^\\([:space:]]+)[[:space:]]*",
    "(?:\\(([<>=!]+)[[:space:]]*([^\\)]+)\\))?",
    "[[:space:]]*$"
  )

  match <- regexec(pattern, x, perl = TRUE)
  pieces <- regmatches(x, match)[[1]]

  if (!length(pieces)) {
    name <- sub("^([[:alnum:].]+).*$", "\\1", x)
    return(list(name = name, op = "", version = "", raw = x))
  }

  list(
    name = pieces[[2]],
    op = if (length(pieces) >= 3L) pieces[[3]] else "",
    version = if (length(pieces) >= 4L) trimws(pieces[[4]]) else "",
    raw = x
  )
}

parse_dependency_field <- function(parent, field, value) {
  if (is.na(value) || !nzchar(trimws(value))) return(NULL)

  entries <- unlist(strsplit(value, ",", fixed = TRUE), use.names = FALSE)
  specs <- lapply(entries, split_dependency_spec)
  specs <- Filter(
    function(spec) {
      !is.null(spec) && nzchar(spec$name) && !identical(spec$name, "R")
    },
    specs
  )

  if (!length(specs)) return(NULL)

  data.frame(
    parent = rep(parent, length(specs)),
    field = rep(field, length(specs)),
    dependency = vapply(specs, `[[`, character(1), "name"),
    op = vapply(specs, `[[`, character(1), "op"),
    required_version = vapply(specs, `[[`, character(1), "version"),
    raw = vapply(specs, `[[`, character(1), "raw"),
    stringsAsFactors = FALSE
  )
}

constraint_parts <- list()
constraint_parents <- intersect(requested_closure, rownames(available))
for (parent in constraint_parents) {
  for (field in hard_types) {
    part <- parse_dependency_field(parent, field, available[parent, field])
    if (!is.null(part) && nrow(part)) {
      constraint_parts[[length(constraint_parts) + 1L]] <- part
    }
  }
}

dependency_constraints <- if (length(constraint_parts)) {
  do.call(rbind, constraint_parts)
} else {
  data.frame(
    parent = character(),
    field = character(),
    dependency = character(),
    op = character(),
    required_version = character(),
    raw = character(),
    stringsAsFactors = FALSE
  )
}

version_requirements <- dependency_constraints[
  nzchar(dependency_constraints$op) &
    nzchar(dependency_constraints$required_version),
  ,
  drop = FALSE
]

write.csv(
  version_requirements,
  file.path(log_dir, "dependency-version-requirements.csv"),
  row.names = FALSE
)

constraints_for <- function(pkg) {
  version_requirements[
    version_requirements$dependency == pkg,
    ,
    drop = FALSE
  ]
}

satisfies_version_predicate <- function(installed, op, required) {
  if (is.na(installed) || !nzchar(installed)) return(FALSE)
  if (is.na(op) || !nzchar(op) || is.na(required) || !nzchar(required)) {
    return(TRUE)
  }

  cmp <- tryCatch(
    utils::compareVersion(installed, required),
    error = function(e) NA_integer_
  )
  if (is.na(cmp)) return(FALSE)

  switch(
    op,
    ">=" = cmp >= 0L,
    ">" = cmp > 0L,
    "<=" = cmp <= 0L,
    "<" = cmp < 0L,
    "==" = cmp == 0L,
    "=" = cmp == 0L,
    "!=" = cmp != 0L,
    FALSE
  )
}

version_satisfies_constraints <- function(pkg, installed) {
  requirements <- constraints_for(pkg)
  if (!nrow(requirements)) return(TRUE)

  all(vapply(
    seq_len(nrow(requirements)),
    function(i) {
      satisfies_version_predicate(
        installed,
        requirements$op[[i]],
        requirements$required_version[[i]]
      )
    },
    logical(1)
  ))
}

constraint_summary <- function(pkg) {
  requirements <- constraints_for(pkg)
  if (!nrow(requirements)) return("")

  paste(unique(sprintf(
    "%s:%s requires %s %s",
    requirements$parent,
    requirements$field,
    requirements$op,
    requirements$required_version
  )), collapse = "; ")
}

# Fail early if dependency parsing or R package-version comparison behaves
# differently than expected on the running R build.  The Rcpp requirement is a
# real-world case used by current lme4 releases.
parser_probe <- split_dependency_spec("Rcpp (>= 1.1.1-1.1)")
stopifnot(
  identical(parser_probe$name, "Rcpp"),
  identical(parser_probe$op, ">="),
  identical(parser_probe$version, "1.1.1-1.1"),
  !satisfies_version_predicate("1.1.1", ">=", "1.1.1-1.1"),
  satisfies_version_predicate("1.1.2", ">=", "1.1.1-1.1")
)

mlog(
  "Recorded ", nrow(version_requirements),
  " hard dependency version constraints."
)

topo_sort <- function(pkgs, deps) {
  remaining <- unique(pkgs)
  ordered <- character()
  cycle_members <- character()

  while (length(remaining)) {
    ready <- remaining[vapply(
      remaining,
      function(p) all(deps[[p]] %in% ordered),
      logical(1)
    )]

    if (!length(ready)) {
      cycle_members <- remaining
      ordered <- c(ordered, sort(remaining))
      break
    }

    ready <- sort(ready)
    ordered <- c(ordered, ready)
    remaining <- setdiff(remaining, ready)
  }

  list(order = ordered, cycle_members = cycle_members)
}

topo <- topo_sort(all_pkgs, direct_hard)
install_order <- topo$order
writeLines(install_order, file.path(log_dir, "install-order.txt"))
writeLines(topo$cycle_members, file.path(log_dir, "dependency-cycle-members.txt"))

edges <- do.call(rbind, lapply(names(direct_hard), function(p) {
  if (!length(direct_hard[[p]])) return(NULL)
  data.frame(package = p, dependency = direct_hard[[p]], stringsAsFactors = FALSE)
}))
if (is.null(edges)) edges <- data.frame(package = character(), dependency = character())
write.csv(edges, file.path(log_dir, "dependency-edges.csv"), row.names = FALSE)

plan <- data.frame(
  sequence = seq_along(install_order),
  pkg = install_order,
  role = unname(roles[install_order]),
  repository_version = unname(available[install_order, "Version"]),
  direct_hard_dependencies = vapply(
    install_order,
    function(p) paste(direct_hard[[p]], collapse = ";"),
    character(1)
  ),
  required_constraints = vapply(
    install_order,
    constraint_summary,
    character(1)
  ),
  stringsAsFactors = FALSE
)
write.csv(plan, file.path(log_dir, "install-plan.csv"), row.names = FALSE)

mlog(
  "Closure contains ", length(all_pkgs), " repository packages; ",
  length(unavailable), " unavailable names; direct Suggests = ",
  include_direct_suggests, "."
)

worker_file <- file.path(log_dir, "install-worker.R")
writeLines(c(
  'args <- commandArgs(TRUE)',
  'if (length(args) < 5L) stop("worker requires pkg, expected version, deploy lib, repo, output dir")',
  'pkg <- args[[1]]',
  'expected_version <- args[[2]]',
  'deploy_lib <- args[[3]]',
  'repo <- args[[4]]',
  'output_dir <- args[[5]]',
  'system_libs <- strsplit(Sys.getenv("SYSTEM_R_LIBS"), .Platform$path.sep, fixed = TRUE)[[1]]',
  'system_libs <- system_libs[nzchar(system_libs)]',
  '.libPaths(unique(c(deploy_lib, system_libs)))',
  'options(repos = c(CRAN = repo), warn = 1, Ncpus = 1L)',
  '',
  '# Reassert the tool path inside the clean worker. nloptr reads CMAKE_BIN',
  '# directly; Arrow and other packages commonly read CMAKE or PATH.',
  'cmake_bin <- Sys.getenv("CMAKE_BIN", unset = "")',
  'if (!nzchar(cmake_bin)) cmake_bin <- Sys.getenv("CMAKE", unset = "")',
  'if (nzchar(cmake_bin) && file.exists(cmake_bin)) {',
  '  cmake_bin <- normalizePath(cmake_bin, mustWork = TRUE)',
  '  Sys.setenv(',
  '    CMAKE_BIN = cmake_bin,',
  '    CMAKE = cmake_bin,',
  '    PATH = paste(dirname(cmake_bin), Sys.getenv("PATH"), sep = .Platform$path.sep)',
  '  )',
  '}',
  'dir.create(deploy_lib, recursive = TRUE, showWarnings = FALSE)',
  'dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)',
  'unlink(file.path(deploy_lib, paste0("00LOCK-", pkg)), recursive = TRUE, force = TRUE)',
  'cat("WORKER_PACKAGE=", pkg, "\\n", sep = "")',
  'cat("WORKER_LIBPATHS=", paste(.libPaths(), collapse = ";"), "\\n", sep = "")',
  'cat("WORKER_EXPECTED_VERSION=", expected_version, "\\n", sep = "")',
  'cat("WORKER_PATH=", Sys.getenv("PATH"), "\\n", sep = "")',
  'cat("WORKER_CMAKE=", Sys.getenv("CMAKE"), "\\n", sep = "")',
  'cat("WORKER_CMAKE_BIN=", Sys.getenv("CMAKE_BIN"), "\\n", sep = "")',
  'cat("WORKER_CMAKE_WHICH=", unname(Sys.which("cmake")), "\\n", sep = "")',
  'if (nzchar(Sys.getenv("CMAKE_BIN")) && file.exists(Sys.getenv("CMAKE_BIN"))) {',
  '  cat("WORKER_CMAKE_VERSION_BEGIN\\n")',
  '  cat(system2(Sys.getenv("CMAKE_BIN"), "--version", stdout = TRUE, stderr = TRUE), sep = "\\n")',
  '  cat("\\nWORKER_CMAKE_VERSION_END\\n")',
  '}',
  'cat("WORKER_PKG_CONFIG_WHICH=", unname(Sys.which("pkg-config")), "\\n", sep = "")',
  'cat("WORKER_PKG_CONFIG_PATH=", Sys.getenv("PKG_CONFIG_PATH"), "\\n", sep = "")',
  'configure_parts <- character()',
  'if (identical(pkg, "nloptr") && nzchar(cmake_bin)) {',
  '  configure_parts <- c(configure_parts, paste0("CMAKE_BIN=", shQuote(cmake_bin)))',
  '}',
  'if (identical(pkg, "xml2")) {',
  '  include_dir <- Sys.getenv("XML2_INCLUDE_DIR", unset = "")',
  '  lib_dir <- Sys.getenv("XML2_LIB_DIR", unset = "")',
  '  if (nzchar(include_dir) && nzchar(lib_dir)) {',
  '    configure_parts <- c(',
  '      configure_parts,',
  '      paste0("INCLUDE_DIR=", shQuote(include_dir)),',
  '      paste0("LIB_DIR=", shQuote(lib_dir))',
  '    )',
  '  }',
  '}',
  'if (identical(pkg, "ragg")) {',
  '  include_dir <- Sys.getenv("RAGG_INCLUDE_DIR", unset = "")',
  '  lib_dir <- Sys.getenv("RAGG_LIB_DIR", unset = "")',
  '  if (nzchar(include_dir) && nzchar(lib_dir)) {',
  '    configure_parts <- c(',
  '      configure_parts,',
  '      paste0("INCLUDE_DIR=", shQuote(include_dir)),',
  '      paste0("LIB_DIR=", shQuote(lib_dir))',
  '    )',
  '  }',
  '}',
  'configure_vars <- paste(configure_parts, collapse = " ")',
  'cat("WORKER_CONFIGURE_VARS=", configure_vars, "\\n", sep = "")',
  'utils::install.packages(',
  '  pkg,',
  '  lib = deploy_lib,',
  '  repos = c(CRAN = repo),',
  '  dependencies = FALSE,',
  '  configure.vars = configure_vars,',
  '  type = "source",',
  '  verbose = TRUE,',
  '  keep_outputs = output_dir,',
  '  destdir = file.path(output_dir, "sources"),',
  '  INSTALL_opts = "--no-clean-on-error"',
  ')',
  'pkg_path <- file.path(deploy_lib, pkg)',
  'desc <- file.path(pkg_path, "DESCRIPTION")',
  'if (!file.exists(desc)) stop("package was not installed into deploy_lib: ", pkg)',
  'got_version <- unname(read.dcf(desc, fields = "Version")[1, "Version"])',
  'if (utils::compareVersion(got_version, expected_version) < 0L) {',
  '  stop("installed version ", got_version, " is older than planned version ", expected_version)',
  '}',
  'ns <- loadNamespace(pkg, lib.loc = unique(c(deploy_lib, system_libs)))',
  'loaded_path <- normalizePath(getNamespaceInfo(ns, "path"), mustWork = FALSE)',
  'expected_path <- normalizePath(pkg_path, mustWork = FALSE)',
  'if (!identical(loaded_path, expected_path)) {',
  '  stop("namespace resolved from unexpected path: ", loaded_path)',
  '}',
  'cat("WORKER_RESULT=OK\\n")',
  'cat("WORKER_VERSION=", got_version, "\\n", sep = "")',
  'cat("WORKER_PACKAGE_PATH=", expected_path, "\\n", sep = "")'
), worker_file)

status <- data.frame(
  sequence = integer(),
  pkg = character(),
  role = character(),
  result = character(),
  expected_version = character(),
  installed_version = character(),
  installed_path = character(),
  exit_status = integer(),
  elapsed_seconds = numeric(),
  blockers = character(),
  package_output = character(),
  driver_output = character(),
  stringsAsFactors = FALSE
)

append_status <- function(row) {
  status <<- rbind(status, row)
  write.csv(status, status_file, row.names = FALSE)
  saveRDS(status, file.path(log_dir, "status.rds"))
}

# Record requested base/recommended packages even though they should not be
# reinstalled into the overlay library.
if (length(base_requested)) {
  for (p in sort(base_requested)) {
    rec <- pkg_record(p, search_libs)
    expected <- if (p %in% rownames(available)) {
      unname(available[p, "Version"])
    } else {
      rec$version
    }
    append_status(data.frame(
      sequence = NA_integer_,
      pkg = p,
      role = if (p %in% targets) "target" else "dependency",
      result = "SYSTEM_BASE_RECOMMENDED",
      expected_version = expected,
      installed_version = rec$version,
      installed_path = rec$path,
      exit_status = 0L,
      elapsed_seconds = 0,
      blockers = "",
      package_output = NA_character_,
      driver_output = NA_character_,
      stringsAsFactors = FALSE
    ))
  }
}

is_resolvable <- function(pkg) {
  rec <- pkg_record(pkg, search_libs)
  rec$present && version_satisfies_constraints(pkg, rec$version)
}

needs_install <- function(pkg, expected_version) {
  local <- pkg_record(pkg, deploy_lib)
  resolved <- pkg_record(pkg, search_libs)

  local_compatible <- local$present &&
    version_satisfies_constraints(pkg, local$version)
  resolved_compatible <- resolved$present &&
    version_satisfies_constraints(pkg, resolved$version)

  switch(
    install_policy,
    "all-to-deploy" = !local_compatible ||
      !version_at_least(local$version, expected_version),
    "missing-only" = !resolved_compatible,
    "missing-or-outdated" = !resolved_compatible ||
      !version_at_least(resolved$version, expected_version)
  )
}

harvest_forensics <- function(pkg) {
  lock_root <- file.path(deploy_lib, paste0("00LOCK-", pkg))
  if (!dir.exists(lock_root)) return(character())

  candidates <- list.files(
    lock_root,
    recursive = TRUE,
    full.names = TRUE,
    all.files = TRUE
  )
  candidates <- candidates[grepl(
    "(config\\.log|CMakeError\\.log|CMakeOutput\\.log|CMakeCache\\.txt|configure$|Makevars$)",
    candidates,
    ignore.case = TRUE
  )]
  candidates <- candidates[file.info(candidates)$isdir %in% FALSE]

  if (!length(candidates)) return(character())

  pkg_dir <- file.path(forensics_dir, pkg)
  dir.create(pkg_dir, recursive = TRUE, showWarnings = FALSE)

  copied <- character()
  for (src in candidates) {
    rel <- substring(src, nchar(lock_root) + 2L)
    dest <- file.path(pkg_dir, gsub("[/\\\\]", "__", rel))
    if (file.copy(src, dest, overwrite = TRUE)) copied <- c(copied, dest)
  }
  copied
}

rscript <- file.path(R.home("bin"), "Rscript")
if (!file.exists(rscript)) stop("Rscript not found: ", rscript)

# On Unix, system2() pastes `env` entries directly into a shell command.
# Values containing spaces therefore must be quoted as the VALUE portion of
# NAME=VALUE (quoting the whole assignment would turn it into a command name).
# Keep package-specific discovery hints out of unrelated workers as well.
env_assignment <- function(name, value) {
  stopifnot(length(name) == 1L, length(value) == 1L)
  if (grepl("[\r\n]", value)) {
    stop("Environment value contains a newline: ", name)
  }
  if (.Platform$OS.type == "windows") {
    paste0(name, "=", value)
  } else {
    paste0(name, "=", shQuote(value))
  }
}

common_build_env <- build_env[
  !names(build_env) %in% c(
    "XML2_INCLUDE_DIR", "XML2_LIB_DIR",
    "RAGG_INCLUDE_DIR", "RAGG_LIB_DIR"
  )
]

child_env_values <- c(
  R_LIBS = paste(search_libs, collapse = .Platform$path.sep),
  R_LIBS_USER = deploy_lib,
  R_LIBS_SITE = paste(system_libs, collapse = .Platform$path.sep),
  SYSTEM_R_LIBS = paste(system_libs, collapse = .Platform$path.sep),
  common_build_env
)

make_child_env <- function(pkg) {
  values <- child_env_values

  if (identical(pkg, "xml2")) {
    xml_names <- intersect(
      c("XML2_INCLUDE_DIR", "XML2_LIB_DIR"),
      names(build_env)
    )
    values <- c(values, build_env[xml_names])
  }

  if (identical(pkg, "ragg")) {
    ragg_names <- intersect(
      c("RAGG_INCLUDE_DIR", "RAGG_LIB_DIR"),
      names(build_env)
    )
    values <- c(values, build_env[ragg_names])
  }

  # Last definition wins if a caller deliberately overrides a common value.
  values <- values[!duplicated(names(values), fromLast = TRUE)]

  vapply(
    seq_along(values),
    function(i) env_assignment(names(values)[[i]], unname(values[[i]])),
    character(1),
    USE.NAMES = FALSE
  )
}

# Preserve the unquoted logical values for diagnostics; the quoted assignments
# passed to system2() are also recorded by the launch preflight below.
write.table(
  data.frame(
    name = names(child_env_values),
    value = unname(child_env_values),
    stringsAsFactors = FALSE
  ),
  file.path(log_dir, "child-environment-values.tsv"),
  sep = "\t",
  quote = TRUE,
  row.names = FALSE
)

# Fail before the package loop if the clean worker cannot be launched. This
# catches malformed environment assignments as a wrapper error rather than
# misclassifying every attempted package as a build failure.
preflight_file <- file.path(log_dir, "worker-launch-preflight.R")
writeLines(c(
  'cat("WORKER_LAUNCH_OK\n")',
  'cat("R_LIBS=", Sys.getenv("R_LIBS"), "\n", sep = "")',
  'cat("PATH=", Sys.getenv("PATH"), "\n", sep = "")',
  'cat("CMAKE_BIN=", Sys.getenv("CMAKE_BIN"), "\n", sep = "")',
  'cat("XML2_INCLUDE_DIR=", Sys.getenv("XML2_INCLUDE_DIR"), "\n", sep = "")',
  'cat("RAGG_INCLUDE_DIR=", Sys.getenv("RAGG_INCLUDE_DIR"), "\n", sep = "")'
), preflight_file)

preflight_variants <- c("__common__", "xml2", "ragg")
for (preflight_pkg in preflight_variants) {
  safe_variant <- gsub("[^A-Za-z0-9_.-]", "_", preflight_pkg)
  preflight_log <- file.path(
    log_dir,
    paste0("worker-launch-preflight-", safe_variant, ".log")
  )

  preflight_status <- suppressWarnings(system2(
    rscript,
    args = c("--vanilla", shQuote(preflight_file)),
    stdout = preflight_log,
    stderr = preflight_log,
    env = make_child_env(preflight_pkg),
    wait = TRUE,
    timeout = 60L
  ))

  if (!identical(as.integer(preflight_status), 0L)) {
    stop(
      "Clean worker launch failed for environment variant ", preflight_pkg,
      " with status ", preflight_status, ". See: ", preflight_log
    )
  }
}

for (i in seq_len(nrow(plan))) {
  p <- plan$pkg[[i]]
  role <- plan$role[[i]]
  expected <- plan$repository_version[[i]]
  package_out <- file.path(log_dir, paste0(p, ".out"))
  driver_out <- file.path(log_dir, paste0(p, ".driver.log"))

  resolved_before <- pkg_record(p, search_libs)
  compatible_before <- resolved_before$present &&
    version_satisfies_constraints(p, resolved_before$version)

  if (!needs_install(p, expected)) {
    rec <- resolved_before
    append_status(data.frame(
      sequence = i,
      pkg = p,
      role = role,
      result = "PRESENT",
      expected_version = expected,
      installed_version = rec$version,
      installed_path = rec$path,
      exit_status = 0L,
      elapsed_seconds = 0,
      blockers = "",
      package_output = package_out,
      driver_output = driver_out,
      stringsAsFactors = FALSE
    ))
    constraint_note <- constraint_summary(p)
    mlog(
      "PRESENT: ", p, " ", rec$version, " at ", rec$path,
      if (nzchar(constraint_note)) paste0(" [compatible: ", constraint_note, "]") else ""
    )
    next
  }

  if (resolved_before$present && !compatible_before) {
    mlog(
      "UPGRADE REQUIRED: ", p, " ", resolved_before$version,
      " at ", resolved_before$path,
      " does not satisfy ", constraint_summary(p)
    )
  }

  blockers <- direct_hard[[p]][!vapply(direct_hard[[p]], is_resolvable, logical(1))]
  if (length(blockers)) {
    append_status(data.frame(
      sequence = i,
      pkg = p,
      role = role,
      result = "BLOCKED",
      expected_version = expected,
      installed_version = NA_character_,
      installed_path = NA_character_,
      exit_status = NA_integer_,
      elapsed_seconds = 0,
      blockers = paste(blockers, collapse = ";"),
      package_output = package_out,
      driver_output = driver_out,
      stringsAsFactors = FALSE
    ))
    mlog("BLOCKED: ", p, " by ", paste(blockers, collapse = ", "))
    next
  }

  mlog("INSTALL start: ", p, " (expected ", expected, ")")
  started <- Sys.time()

  exit_status <- tryCatch(
    system2(
      rscript,
      args = c(
        "--vanilla",
        shQuote(worker_file),
        shQuote(p),
        shQuote(expected),
        shQuote(deploy_lib),
        shQuote(repo),
        shQuote(log_dir)
      ),
      stdout = driver_out,
      stderr = driver_out,
      env = make_child_env(p),
      wait = TRUE,
      timeout = install_timeout
    ),
    error = function(e) {
      cat("PARENT_SYSTEM2_ERROR: ", conditionMessage(e), "\n", file = driver_out, append = TRUE)
      127L
    }
  )

  elapsed <- as.numeric(difftime(Sys.time(), started, units = "secs"))
  rec <- pkg_record(p, deploy_lib)
  ok <- identical(as.integer(exit_status), 0L) &&
    rec$present && version_at_least(rec$version, expected)

  if (!ok) harvest_forensics(p)

  append_status(data.frame(
    sequence = i,
    pkg = p,
    role = role,
    result = if (ok) "OK" else "FAILED",
    expected_version = expected,
    installed_version = rec$version,
    installed_path = rec$path,
    exit_status = as.integer(exit_status),
    elapsed_seconds = elapsed,
    blockers = "",
    package_output = package_out,
    driver_output = driver_out,
    stringsAsFactors = FALSE
  ))

  mlog(
    if (ok) "INSTALL OK: " else "INSTALL FAIL: ",
    p,
    " exit=", exit_status,
    " elapsed=", round(elapsed, 1), "s"
  )
}

# Validate requested targets in fresh Rscript --vanilla processes. A package
# recorded as PRESENT has only been located on disk; this separate check proves
# that its namespace and compiled libraries actually load in the final search
# path without relying on namespaces already resident in the notebook kernel.
validation_worker <- file.path(log_dir, "validate-target-worker.R")
writeLines(c(
  'args <- commandArgs(TRUE)',
  'if (length(args) < 2L) stop("validation worker requires pkg and deploy lib")',
  'pkg <- args[[1]]',
  'deploy_lib <- args[[2]]',
  'system_libs <- args[-c(1L, 2L)]',
  '.libPaths(unique(c(deploy_lib, system_libs)))',
  'ns <- loadNamespace(pkg, lib.loc = .libPaths())',
  'pkg_path <- normalizePath(getNamespaceInfo(ns, "path"), mustWork = FALSE)',
  'pkg_version <- as.character(utils::packageVersion(pkg, lib.loc = .libPaths()))',
  'cat("VALIDATION_RESULT=OK\\n")',
  'cat("VALIDATION_PACKAGE=", pkg, "\\n", sep = "")',
  'cat("VALIDATION_VERSION=", pkg_version, "\\n", sep = "")',
  'cat("VALIDATION_PATH=", pkg_path, "\\n", sep = "")'
), validation_worker)

target_validation <- data.frame(
  pkg = targets,
  load_ok = FALSE,
  exit_status = NA_integer_,
  version = NA_character_,
  path = NA_character_,
  log = NA_character_,
  stringsAsFactors = FALSE
)

for (j in seq_along(targets)) {
  p <- targets[[j]]
  validation_log <- file.path(log_dir, paste0(p, ".validation.log"))
  validation_status <- suppressWarnings(system2(
    rscript,
    args = c(
      "--vanilla",
      shQuote(validation_worker),
      shQuote(p),
      shQuote(deploy_lib),
      vapply(system_libs, shQuote, character(1))
    ),
    stdout = validation_log,
    stderr = validation_log,
    env = make_child_env("__common__"),
    wait = TRUE,
    timeout = 300L
  ))

  lines <- safe_lines(validation_log)
  get_marker <- function(prefix) {
    hit <- grep(paste0("^", prefix), lines, value = TRUE)
    if (length(hit)) sub(paste0("^", prefix), "", tail(hit, 1L)) else NA_character_
  }

  target_validation$exit_status[[j]] <- as.integer(validation_status)
  target_validation$load_ok[[j]] <- identical(as.integer(validation_status), 0L) &&
    any(lines == "VALIDATION_RESULT=OK")
  target_validation$version[[j]] <- get_marker("VALIDATION_VERSION=")
  target_validation$path[[j]] <- get_marker("VALIDATION_PATH=")
  target_validation$log[[j]] <- validation_log
}

write.csv(
  target_validation,
  file.path(log_dir, "target-validation.csv"),
  row.names = FALSE
)

# Add unresolved repository names to the machine-readable status table.
if (length(unavailable)) {
  for (p in sort(unavailable)) {
    append_status(data.frame(
      sequence = NA_integer_,
      pkg = p,
      role = if (p %in% targets) "target" else "dependency",
      result = "UNAVAILABLE",
      expected_version = NA_character_,
      installed_version = NA_character_,
      installed_path = NA_character_,
      exit_status = NA_integer_,
      elapsed_seconds = 0,
      blockers = "not present in repository metadata",
      package_output = NA_character_,
      driver_output = NA_character_,
      stringsAsFactors = FALSE
    ))
  }
}

failed <- status$pkg[status$result == "FAILED"]
blocked <- status$pkg[status$result == "BLOCKED"]
blocked_native <- status$pkg[status$result == "BLOCKED_NATIVE"]
unavailable_status <- status$pkg[status$result == "UNAVAILABLE"]
validation_failed <- target_validation$pkg[!target_validation$load_ok]

mlog("==================== SUMMARY ====================")
mlog(
  "OK=", sum(status$result == "OK", na.rm = TRUE),
  " PRESENT=", sum(status$result == "PRESENT", na.rm = TRUE),
  " SYSTEM_BASE_RECOMMENDED=", sum(status$result == "SYSTEM_BASE_RECOMMENDED", na.rm = TRUE),
  " FAILED=", length(failed),
  " BLOCKED=", length(blocked),
  " BLOCKED_NATIVE=", length(blocked_native),
  " UNAVAILABLE=", length(unavailable_status),
  " TARGET_LOAD_OK=", sum(target_validation$load_ok),
  " TARGET_LOAD_FAILED=", length(validation_failed)
)
if (length(failed)) mlog("FAILED: ", paste(failed, collapse = ", "))
if (length(blocked)) mlog("BLOCKED: ", paste(blocked, collapse = ", "))
if (length(blocked_native)) mlog("BLOCKED_NATIVE: ", paste(blocked_native, collapse = ", "))
if (length(unavailable_status)) mlog("UNAVAILABLE: ", paste(unavailable_status, collapse = ", "))
if (length(validation_failed)) {
  mlog("TARGET LOAD FAILED: ", paste(validation_failed, collapse = ", "))
} else {
  mlog("All requested targets loaded successfully in fresh R processes.")
}

error_pattern <- paste(
  c(
    "ERROR:", "Error in", "Execution halted", "non-zero exit status",
    "configuration failed", "configure: error", "CMake Error", "FAILED",
    "fatal error", "not found", "cannot find", "No such file",
    "No package .* found", "pkg-config", "unable to load", "undefined symbol",
    "GLIBC", "GLIBCXX", "relocation", "collect2:", "ld returned",
    "namespace .* required", "dependency .* is not available", "udunits",
    "gdal", "geos", "proj", "libxml", "fontconfig", "cairo", "openssl",
    "freetype", "harfbuzz", "fribidi"
  ),
  collapse = "|"
)

diag_file <- file.path(log_dir, "failure-causes.txt")
diag_con <- file(diag_file, open = "wt")

for (p in unique(c(failed, blocked, blocked_native, unavailable_status))) {
  writeLines(paste0("### ", p), diag_con)

  if (p %in% blocked) {
    row <- status[status$pkg == p & status$result == "BLOCKED", , drop = FALSE]
    writeLines(paste0("BLOCKED BY: ", row$blockers[[1]]), diag_con)
  }

  if (p %in% blocked_native) {
    row <- status[status$pkg == p & status$result == "BLOCKED_NATIVE", , drop = FALSE]
    writeLines(paste0("BLOCKED BY NATIVE REQUIREMENTS: ", row$blockers[[1]]), diag_con)
  }

  if (p %in% unavailable_status) {
    writeLines("UNAVAILABLE IN REPOSITORY METADATA", diag_con)
  }

  log_candidates <- c(
    file.path(log_dir, paste0(p, ".out")),
    file.path(log_dir, paste0(p, ".driver.log")),
    list.files(file.path(forensics_dir, p), recursive = TRUE, full.names = TRUE)
  )
  log_candidates <- unique(log_candidates[file.exists(log_candidates)])

  lines <- strip_ansi(unlist(lapply(log_candidates, safe_lines), use.names = FALSE))
  hits <- grep(error_pattern, lines, ignore.case = TRUE, value = TRUE, perl = TRUE)

  if (length(hits)) {
    writeLines(utils::tail(unique(hits), 60), diag_con)
  } else if (length(lines)) {
    writeLines(utils::tail(lines, 40), diag_con)
  } else {
    writeLines("No package log was available.", diag_con)
  }
  writeLines("", diag_con)
}

close(diag_con)

# Record checksums for retained source archives when sha256sum is available.
source_archives <- list.files(source_dir, pattern = "\\.tar\\.gz$", full.names = TRUE)
sha256 <- unname(Sys.which("sha256sum"))
if (length(source_archives) && nzchar(sha256)) {
  checksum_lines <- unlist(lapply(source_archives, function(f) {
    system2(sha256, shQuote(f), stdout = TRUE, stderr = TRUE)
  }), use.names = FALSE)
  writeLines(checksum_lines, file.path(log_dir, "SHA256SUMS"))
}

mlog("Status table: ", status_file)
mlog("Target validation: ", file.path(log_dir, "target-validation.csv"))
mlog("Failure summary: ", diag_file)
mlog("All logs and source archives: ", log_dir)

cat("\nUseful read-back commands:\n")
cat("  cat ", shQuote(master_log), "\n", sep = "")
cat("  cat ", shQuote(status_file), "\n", sep = "")
cat("  cat ", shQuote(file.path(log_dir, "target-validation.csv")), "\n", sep = "")
cat("  cat ", shQuote(diag_file), "\n", sep = "")

# Set only after the complete installer and its target validation finish.
installer_completed <- TRUE


In [ ]:
# OPTIONAL: persist this run's deployment library for future R sessions.
# Run only after reviewing the installer summary. This replaces the home
# R_LIBS_USER setting, backs up an existing file, and preserves other lines.
# Skip this cell if your administrator manages R startup configuration.

local({
  if (!exists("installer_completed", inherits = TRUE) ||
      !isTRUE(installer_completed) ||
      !exists("deploy_lib", inherits = TRUE) ||
      !exists("target_validation", inherits = TRUE) ||
      !nrow(target_validation) ||
      !all(target_validation$load_ok)) {
    stop("Complete the installer with all requested targets loading successfully before persisting the library.")
  }

  renviron <- path.expand("~/.Renviron")
  if (nzchar(Sys.getenv("R_ENVIRON_USER", unset = ""))) {
    stop("R_ENVIRON_USER selects another startup file; configure R_LIBS_USER there manually.")
  }
  project_renviron <- file.path(getwd(), ".Renviron")
  if (file.exists(project_renviron) &&
      !identical(normalizePath(project_renviron, mustWork = TRUE),
                 normalizePath(renviron, mustWork = FALSE))) {
    stop("A project .Renviron shadows the home file; configure R_LIBS_USER there manually.")
  }

  persist_lib <- normalizePath(deploy_lib, mustWork = TRUE)
  # Keep the default template portable across R versions/platforms; preserve
  # a custom deployment library exactly for this R version and platform.
  if (identical(persist_lib, normalizePath(default_deploy_lib, mustWork = FALSE))) {
    entry <- 'R_LIBS_USER="${HOME}/R/library/%v/%p"'
  } else {
    # R startup expands ${...} and library-path % tokens. Refuse ambiguous
    # paths instead of writing a value that would select a different library.
    special <- c("\r", "\n", '"', "\\", "${", "%", .Platform$path.sep)
    if (any(vapply(special, function(x) grepl(x, persist_lib, fixed = TRUE), logical(1)))) {
      stop("Custom deployment path contains startup-file special characters; configure R_LIBS_USER manually.")
    }
    entry <- paste0('R_LIBS_USER="', persist_lib, '"')
  }

  existing <- if (file.exists(renviron)) {
    readLines(renviron, warn = FALSE)
  } else {
    character()
  }

  if (file.exists(renviron)) {
    backup <- tempfile(
      pattern = paste0(".Renviron.bak-", format(Sys.time(), "%Y%m%d-%H%M%S"), "-"),
      tmpdir = dirname(renviron)
    )
    if (!isTRUE(file.copy(renviron, backup, overwrite = FALSE, copy.mode = TRUE))) {
      stop("Could not back up ", renviron, "; no startup change was written.")
    }
    cat("Backup written to:\n", backup, "\n")
  }

  existing <- existing[!grepl("^\\s*R_LIBS_USER\\s*=", existing)]
  staged <- tempfile(pattern = ".Renviron-", tmpdir = dirname(renviron))
  tryCatch({
    writeLines(c(existing, entry), staged)
    mode <- if (file.exists(renviron)) file.info(renviron)$mode else "0600"
    if (!isTRUE(Sys.chmod(staged, mode = mode))) {
      stop("Could not preserve startup-file permissions; no startup change was written.")
    }
    if (!isTRUE(file.rename(staged, renviron))) {
      stop("Could not replace ", renviron, "; the previous file was retained.")
    }
  }, finally = {
    if (file.exists(staged)) unlink(staged)
  })

  # Do not print the complete .Renviron: unrelated lines may contain secrets.
  cat("Updated:", renviron, "\n", entry, "\n", sep = "")
  cat("Restart the R kernel, then verify Sys.getenv('R_LIBS_USER') and .libPaths().\n")
})
